In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import cv2
import sys
import os


home = Path.home()
print(home)
#sys.path.append(os.path.join(home,"analysis/multiplex/raw_counter/src"))
#sys.path.append(os.path.join(Path.cwd(), 'src'))
if "ivana" in str(home):
    home = "C:/Users/ivana"
    str_path =   "/Dropbox/Ivana Ducrey/Doctorado- Drosophila melanogaster/Berlin Lisa/analysis/behavior/raw_counter/src"
    path2add = str(home) + str_path
    home_dir = str(home) + "/Dropbox/Ivana Ducrey/Doctorado- Drosophila melanogaster/Berlin Lisa/data/multiplex"
else:
    path2add = str(home) + "/analysis/behavior/raw_counter/src"
    #/Users/gerard/analysis/behavior/raw_counter/src/counter_functions.py
    home_dir = os.path.join(home, "data/multiplex")
print(path2add)
sys.path.append(path2add)
from counter_functions import *


#home_dir = Path.home() / "data" / "multiplex"
print(home_dir)






C:\Users\ivana
C:/Users/ivana/Dropbox/Ivana Ducrey/Doctorado- Drosophila melanogaster/Berlin Lisa/analysis/behavior/raw_counter/src
C:/Users/ivana/Dropbox/Ivana Ducrey/Doctorado- Drosophila melanogaster/Berlin Lisa/data/multiplex


In [2]:


date = '11_05_2026'
movies2check = [
               0, 
                1,
                ]

loc_training_oddors = ['down', 'up'] # up = MCH, down = Octanol
#movie_locs = [f"{home_dir}/{date}/{date}_movie_{num}.mp4" for num in movies2check]
paired_exp = True
PIs =[]
PIs_final = []

do_not_show = True



In [5]:
for i, movie_number in enumerate(movies2check):
    # create directory to store frames etc..
    out_dir = f"{home_dir}/{date}/{date}_movie_{movie_number}"
    os.makedirs(out_dir, exist_ok=True)
    movie_loc = f"{home_dir}/{date}/{date}_movie_{movie_number}.mp4"
    print(movie_loc)
    # only keeps the frames used below (old: frames = xtract_frames(movie_loc), ~9 GB per movie)
    kept_frames, last_frames = xtract_needed_frames(movie_loc)
    # selected frames/ 
    last = np.median(np.stack(last_frames), axis=0).astype(np.uint8)   # ~0.7 s
    background_frame = background(kept_frames)
    del kept_frames, last_frames
    background_aligned = align_background(background_frame, last)

    #localize flies:
    diff, mask_flies, overlay = loc_flies(background_aligned, last, threshold=15)
    
    
    # check or draw ROIs:
    roi_file = f"{out_dir}/{date}_rois_{movie_number}.json"
    print(f"{out_dir}/{date}_overlay_{movie_number}.png")
    cv2.imwrite(f"{out_dir}/{date}_overlay_{movie_number}.png", overlay)

   
    rois, cid = load_or_draw_rois(overlay, roi_file, update_date_rois=True, do_not_show = do_not_show)
    plt.close('all')   # free the ROI figures
    
    
    #count flies
    overlay, counts, pi_maze = count_flies(mask_flies, overlay, rois, loc_training_oddors[i])
    PIs.append(pi_maze)
    
    # store images
    cv2.imwrite(f"{out_dir}/{date}_last_{movie_number}.png", last)
    print("background_aligned")
    print(f"{out_dir}/{date}_background_{movie_number}.png")
    cv2.imwrite(f"{out_dir}/{date}_background_{movie_number}.png", background_aligned)

    cv2.imwrite(f"{out_dir}/{date}_diff_{movie_number}.png", cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX))
    cv2.imwrite(f"{out_dir}/{date}_mask_flies_{movie_number}.png", mask_flies)
    print("*******************  MOVIE DONE  ****************")
if paired_exp:
    for maze, val in enumerate(PIs[0]):
        if ~np.isnan(PIs[0][maze]):
            print((PIs[0][maze] + PIs[1][maze])/2)

            

    


C:/Users/ivana/Dropbox/Ivana Ducrey/Doctorado- Drosophila melanogaster/Berlin Lisa/data/multiplex/11_05_2026/11_05_2026_movie_0.mp4
1736 frames
region 0,0: shift dx=-0.8 dy=-2.1 px (response 0.86)
region 0,1: shift dx=+0.0 dy=-2.3 px (response 0.69)
region 0,2: shift dx=+0.8 dy=-2.2 px (response 0.75)
region 0,3: shift dx=+1.5 dy=-2.1 px (response 0.62)
region 1,0: shift dx=-0.2 dy=-0.4 px (response 0.82)
region 1,1: shift dx=+0.3 dy=-0.5 px (response 0.69)
region 1,2: shift dx=+0.8 dy=-0.7 px (response 0.91)
region 1,3: shift dx=+1.4 dy=-0.8 px (response 0.76)
Otsu threshold: 26.0
77 blobs, 68 with area >= 60 px
[np.int32(60), np.int32(71), np.int32(81), np.int32(82), np.int32(90), np.int32(92), np.int32(93), np.int32(95), np.int32(96), np.int32(97), np.int32(98), np.int32(98), np.int32(111), np.int32(118), np.int32(119), np.int32(132), np.int32(135), np.int32(140), np.int32(146), np.int32(149), np.int32(155), np.int32(158), np.int32(162), np.int32(166), np.int32(175), np.int32(179), 

In [8]:
results = []
if paired_exp:
    for maze, val in enumerate(PIs[0]):
        if ~np.isnan(PIs[0][maze]):
            value= (PIs[0][maze] + PIs[1][maze])/2
            print(value)
            results.append({
                "Maze": maze,
                "Mean_PI": value
            })

import pandas as pd
results_file = os.path.join(home_dir, date, "mean_PI.xlsx")

pd.DataFrame(results).to_excel(results_file, index=False)


-0.04
-0.6923076923076923
-0.23529411764705882
